# Color scale helper

Small helper notebook to generate/preview colors for a categorical trait (e.g. `BC`, `DE`, `CTERM` epitopes in `resources/colors.tsv`) and produce TSV lines ready to paste in.

Workflow:
1. List the trait values you need colors for (existing ones you want to keep + new ones to add).
2. The notebook pulls a color scale from `resources/color_schemes.tsv` (same file/logic `scripts/assign-colors.py` uses: line *N* of that file is a scale of *N* distinct colors) sized to the total number of values for the trait, and assigns it in order.
3. Preview swatches inline, tweak/reorder as desired.
4. Print TSV lines to paste into `resources/colors.tsv`.

In [4]:
from pathlib import Path

from IPython.display import HTML, display

COLORS_TSV = Path("../resources/colors.tsv")
COLOR_SCHEMES_TSV = Path("../resources/color_schemes.tsv")


def load_color_schemes():
    """schemes[n] -> list of n hex colors, taken from line n of resources/color_schemes.tsv
    (same file scripts/assign-colors.py draws its color scales from)."""
    schemes = {}
    for i, line in enumerate(COLOR_SCHEMES_TSV.read_text().splitlines(), start=1):
        colors = line.strip().split("\t")
        if colors and colors[0]:
            schemes[i] = colors
    return schemes


COLOR_SCHEMES = load_color_schemes()

In [5]:
def swatch(colors, labels=None):
    """Render a row of color swatches with hex codes (and optional labels) in the notebook."""
    labels = labels or [""] * len(colors)
    cells = "".join(
        f'<div style="display:inline-block;text-align:center;margin:4px;">'
        f'<div style="width:70px;height:40px;background:{c};border:1px solid #999;"></div>'
        f'<div style="font-size:11px;font-family:monospace;">{c}<br>{l}</div>'
        f'</div>'
        for c, l in zip(colors, labels)
    )
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;">{cells}</div>'))


def used_colors_for_trait(trait):
    """Return {value: color} already present for `trait` in resources/colors.tsv."""
    existing = {}
    for line in COLORS_TSV.read_text().splitlines():
        parts = line.strip().split("\t")
        if len(parts) == 3 and parts[0] == trait:
            existing[parts[1]] = parts[2]
    return existing


def color_scale(n):
    """n colors from resources/color_schemes.tsv, mirroring scripts/assign-colors.py:
    use the line with exactly n colors, or concatenate/reuse scales if n exceeds the
    largest available scheme."""
    max_n = max(COLOR_SCHEMES)
    if n <= max_n:
        return list(COLOR_SCHEMES[n])

    colors = []
    remaining = n
    while remaining > 0:
        take = min(remaining, max_n)
        colors += COLOR_SCHEMES[take]
        remaining -= take
    return colors


def assign_colors(trait, new_values, keep_existing=True, other_color="#a9a9a9"):
    """
    Build a full value->color mapping for `trait`.

    - Existing values for the trait keep their current color (if keep_existing).
    - All values (existing kept ones + new ones, excluding 'other') are (re-)assigned
      colors from the color_schemes.tsv scale sized to their total count, in order,
      so the new values get colors from the same scale rather than clashing hues.
    - 'other' (if present/added) always gets `other_color` and sorts last.
    """
    existing = used_colors_for_trait(trait) if keep_existing else {}
    existing_values = [v for v in existing if v != "other"]
    ordered_values = existing_values + [v for v in new_values if v not in existing and v != "other"]

    colors = color_scale(len(ordered_values))
    mapping = dict(zip(ordered_values, colors))

    if "other" in new_values or "other" in existing:
        mapping["other"] = other_color

    return mapping

## Example: adding new BC epitope values

Edit `trait` / `new_values` below for whatever you're adding next.

In [ ]:
trait = "BC"
new_values = []
new_values = [v.upper() for v in new_values]  # values to add, uppercase amino-acid codes

mapping = assign_colors(trait, new_values)
swatch(list(mapping.values()), list(mapping.keys()))

In [14]:
# TSV lines to paste into resources/colors.tsv (replace the existing trait block, `other` line last)
for value, color in mapping.items():
    if value != "other":
        print(f"{trait}\t{value}\t{color}")
if "other" in mapping:
    print(f"{trait}\tother\t{mapping['other']}")

CTERM	KEKANDVTHNIT	#5E1D9D
CTERM	KERASDVTHNIT	#462EB9
CTERM	KERASDVTHNIN	#3F4CCB
CTERM	KERANDVTHNIN	#416CCE
CTERM	KERANNVTHNIT	#4887C6
CTERM	KDRANDVTHNIT	#539CB3
CTERM	KKRANDVTHNIT	#62AB9C
CTERM	KERANDVTHNIT	#74B582
CTERM	KERANGVTHNIT	#89BB6B
CTERM	RDRANDVTHNIT	#A0BE59
CTERM	RERANDVTHNIT	#B7BD4B
CTERM	KERANEITHDLN	#CCB742
CTERM	KERANDITHDIN	#DDAA3C
CTERM	KERANEITHDIN	#E69537
CTERM	KERANEVTHDIN	#E67631
CTERM	RERANEVTHDIT	#E14F2A
CTERM	many x	#DB2823
CTERM	other	#a9a9a9


## Write back into resources/colors.tsv

Replaces the whole `trait` block in-place (all non-`other` lines get the new
nextstrain-scale colors, `other` line/color and everything outside the block are left untouched).

In [ ]:
def write_back(trait, mapping, dry_run=True):
    """Replace all `trait\\t...` lines in resources/colors.tsv with `mapping`,
    in mapping's order, 'other' last. Lines for other traits, blank separator
    lines, etc. are left untouched. Set dry_run=False to actually write the file."""
    lines = COLORS_TSV.read_text().splitlines()

    new_block = [f"{trait}\t{value}\t{color}" for value, color in mapping.items() if value != "other"]
    if "other" in mapping:
        new_block.append(f"{trait}\tother\t{mapping['other']}")

    out = []
    inserted = False
    for line in lines:
        if line.split("\t")[0] == trait:
            if not inserted:
                out.extend(new_block)
                inserted = True
            # skip original lines for this trait; they've been replaced
        else:
            out.append(line)
    if not inserted:
        out.extend(new_block)

    text = "\n".join(out) + "\n"
    if dry_run:
        print(text)
    else:
        COLORS_TSV.write_text(text)
    return text


write_back(trait, mapping, dry_run=True)  # inspect, then re-run with dry_run=False to apply